# 🏋️ Regex Practice Gym: 100 exercises

A companion to *Regex from Scratch*. Same checkers, same ✅/❌ feedback, but now it's **100 fresh exercises** with no lessons in between. Repetition is how patterns become automatic.

| Tier | Exercises | What to expect |
|---|---|---|
| 🟢 **Warm-up** | 1–25 | One or two ideas: literals, `\d`, `\w`, classes, `+ * ? {n,m}`, simple validation |
| 🟡 **Core** | 26–60 | Realistic formats, anchors, groups, `sub`/`split`, lookarounds |
| 🟠 **Challenge** | 61–85 | Multi-part formats, tricky ranges, named groups, text mining |
| 🔴 **Boss** | 86–100 | Assembled patterns, tokenizers, flags, callbacks, pandas |

**How to use it**
* Run the setup cell, then work top to bottom. Or jump around. Each exercise stands alone (a few reuse a variable that's defined in the same cell).
* Fill in `pattern = r""` (and `repl`/`flags` where shown). The checker tells you what failed.
* Read the failing cases. They're usually the clue ("oh, I forgot `\b`").
* Stuck for more than a few minutes? Check the hint, then the answer key at the bottom, then **redo it from memory the next day**.

> Quick reminders: raw strings `r"..."` · validate with `fullmatch` · `findall` returns *groups* if you capture, so use `(?:...)` for grouping only · build big patterns from small tested pieces.

## Setup (run this first)

In [ ]:
import re

# ---------- demo helper ----------
def try_it(pattern, strings, flags=0):
    """Demo helper: show which strings match `pattern` IN FULL."""
    for s in strings:
        ok = re.fullmatch(pattern, s, flags) is not None
        print("✔ match   " if ok else "✘ no match", repr(s))

# ---------- checkers ----------
def _report(problems, total):
    if not problems:
        print(f"✅ Nice! All {total} checks passed.")
    else:
        print(f"❌ {len(problems)} of {total} checks failed:")
        for p in problems:
            print(p)

def _compile(pattern, flags=0):
    try:
        return re.compile(pattern, flags)
    except re.error as e:
        print(f"❌ That isn't a valid regex yet: {e}")
        return None

def check_fullmatch(pattern, should_match, should_not_match, flags=0, label=None):
    """Validation: the pattern must match the ENTIRE string (re.fullmatch)."""
    if label:
        print(f"── {label} ──")
    rx = _compile(pattern, flags)
    if rx is None:
        return
    problems = []
    for s in should_match:
        if not rx.fullmatch(s):
            problems.append(f"   should MATCH but didn't     -> {s!r}")
    for s in should_not_match:
        if rx.fullmatch(s):
            problems.append(f"   should be REJECTED but matched -> {s!r}")
    _report(problems, len(should_match) + len(should_not_match))

def check_search(pattern, should_find, should_not_find, flags=0):
    """The pattern must be FOUND somewhere in each 'good' string (re.search)."""
    rx = _compile(pattern, flags)
    if rx is None:
        return
    problems = []
    for s in should_find:
        if not rx.search(s):
            problems.append(f"   should be FOUND but wasn't   -> {s!r}")
    for s in should_not_find:
        if rx.search(s):
            problems.append(f"   should NOT be found but was  -> {s!r}")
    _report(problems, len(should_find) + len(should_not_find))

def _compare(got, expected, what):
    if got == expected:
        _report([], 1)
    else:
        _report([f"   expected {expected!r}\n   got      {got!r}"], 1)

def check_findall(pattern, text, expected, flags=0):
    rx = _compile(pattern, flags)
    if rx is not None:
        _compare(rx.findall(text), expected, "findall")

def check_sub(pattern, repl, text, expected, flags=0):
    rx = _compile(pattern, flags)
    if rx is not None:
        _compare(rx.sub(repl, text), expected, "sub")

def check_split(pattern, text, expected, flags=0):
    rx = _compile(pattern, flags)
    if rx is not None:
        _compare(rx.split(text), expected, "split")

def check_groups(pattern, text, expected, flags=0):
    """Compare m.groups() of re.search(pattern, text) with a tuple."""
    rx = _compile(pattern, flags)
    if rx is None:
        return
    m = rx.search(text)
    _compare(m.groups() if m else None, expected, "groups")

def check_named(pattern, text, expected, flags=0):
    """Compare m.groupdict() of re.search(pattern, text) with a dict."""
    rx = _compile(pattern, flags)
    if rx is None:
        return
    m = rx.search(text)
    _compare(m.groupdict() if m else None, expected, "groupdict")

def check_tokens(pattern, text, expected, flags=0):
    """Compare [(group_name, matched_text), ...] from re.finditer."""
    rx = _compile(pattern, flags)
    if rx is None:
        return
    got = [(m.lastgroup, m.group()) for m in rx.finditer(text)]
    _compare(got, expected, "tokens")

print("Setup complete ✔")

---
# 🟢 Warm-up (1–25)

In [ ]:
# ✏️ #1 — Find every digit
pattern = r""   # <- your pattern here
check_findall(pattern, 'Locker 7, shelf 3, box 12', ['7', '3', '1', '2'])

In [ ]:
# ✏️ #2 — Find every whole number
pattern = r""   # <- your pattern here
check_findall(pattern, '3 cats, 15 dogs, 200 fish', ['3', '15', '200'])

In [ ]:
# ✏️ #3 — Literal match: "the" also hits inside other words
pattern = r""   # <- your pattern here
check_findall(pattern, 'the other one is the best', ['the', 'the', 'the'])

In [ ]:
# ✏️ #4 — Match the literal text $5.00 (escape the metacharacters!)
pattern = r""   # <- your pattern here
check_findall(pattern, 'Cost: $5.00 or $5x00', ['$5.00'])

In [ ]:
# ✏️ #5 — Wildcard: b, any one character, t
pattern = r""   # <- your pattern here
check_findall(pattern, 'bat bet bit bt boot', ['bat', 'bet', 'bit'])

In [ ]:
# ✏️ #6 — A 4-digit PIN
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['1234', '0000'],
    should_not_match=['123', '12345', '12a4', ''],
)

In [ ]:
# ✏️ #7 — A two-letter state code (capitals only)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['NY', 'CA'],
    should_not_match=['ny', 'N', 'NYC', 'N1'],
)

In [ ]:
# ✏️ #8 — Find capitalised words (capital, then lowercase letters)
pattern = r""   # <- your pattern here
check_findall(pattern, 'Alice met Bob in Paris', ['Alice', 'Bob', 'Paris'])

In [ ]:
# ✏️ #9 — ZIP or ZIP+4: 12345 or 12345-6789
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['12345', '12345-6789'],
    should_not_match=['1234', '12345-', '12345-678', '123456'],
)

In [ ]:
# ✏️ #10 — Find every run of lowercase letters
pattern = r""   # <- your pattern here
check_findall(pattern, 'hello World foo Bar', ['hello', 'orld', 'foo', 'ar'])

In [ ]:
# ✏️ #11 — Find every run of whitespace
pattern = r""   # <- your pattern here
check_findall(pattern, 'a b  c\td', [' ', '  ', '\t'])

In [ ]:
# ✏️ #12 — Find every non-whitespace token
pattern = r""   # <- your pattern here
check_findall(pattern, '  hello   big world ', ['hello', 'big', 'world'])

In [ ]:
# ✏️ #13 — yes/no answers: y, yes, n or no (lowercase)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['y', 'yes', 'n', 'no'],
    should_not_match=['Yes', 'maybe', 'ye', 'yess', 'nop', ''],
)

In [ ]:
# ✏️ #14 — Words ending in "ing"
pattern = r""   # <- your pattern here
check_findall(pattern, 'He was running, jumping and singing.', ['running', 'jumping', 'singing'])

In [ ]:
# ✏️ #15 — A hex string (one or more hex digits)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['ff', 'A9', '0'],
    should_not_match=['', 'g', '12 3'],
)

In [ ]:
# ✏️ #16 — Find integers, including negatives
pattern = r""   # <- your pattern here
check_findall(pattern, 'temps: -5, 12, -30, 7', ['-5', '12', '-30', '7'])

In [ ]:
# ✏️ #17 — Find decimal numbers (digits, dot, digits)
pattern = r""   # <- your pattern here
check_findall(pattern, 'pi 3.14, e 2.718, 42, 0.5', ['3.14', '2.718', '0.5'])

In [ ]:
# ✏️ #18 — Time shape HH:MM (two digits, colon, two digits)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['09:30', '23:59'],
    should_not_match=['9:30', '09:3', '09-30'],
)

In [ ]:
# ✏️ #19 — Find hashtags (# followed by word characters)
pattern = r""   # <- your pattern here
check_findall(pattern, 'Loving #python and #regex101 today!', ['#python', '#regex101'])

In [ ]:
# ✏️ #20 — Find @mentions
pattern = r""   # <- your pattern here
check_findall(pattern, 'ping @alice and @bob_99 now', ['@alice', '@bob_99'])

In [ ]:
# ✏️ #21 — A binary string (only 0s and 1s, at least one)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['0', '101', '1111'],
    should_not_match=['', '102', '10 1'],
)

In [ ]:
# ✏️ #22 — Mask every digit with X
pattern = r""   # <- your pattern here
repl = ""      # <- replacement
check_sub(pattern, repl, 'Pin 4821', 'Pin XXXX')

In [ ]:
# ✏️ #23 — Split on a comma plus optional spaces
pattern = r""   # <- your pattern here
check_split(pattern, 'a, b,c,   d', ['a', 'b', 'c', 'd'])

In [ ]:
# ✏️ #24 — Starts with a capital letter
pattern = r""   # <- your pattern here
check_search(
    pattern,
    should_find=['Hello', 'A'],
    should_not_find=['hello', ' Hello', '1abc'],
)

In [ ]:
# ✏️ #25 — Ends with a question mark
pattern = r""   # <- your pattern here
check_search(
    pattern,
    should_find=['Why?', '?'],
    should_not_find=['Why', 'Why? ', '?x'],
)

---
# 🟡 Core (26–60)

In [ ]:
# ✏️ #26 — Extract phone numbers shaped like 555-123-4567
pattern = r""   # <- your pattern here
check_findall(pattern, 'Call 555-123-4567 or 555-987-6543, not 55-123-4567.', ['555-123-4567', '555-987-6543'])

In [ ]:
# ✏️ #27 — A date shaped DD/MM/YYYY
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['12/11/2024', '01/01/1999'],
    should_not_match=['1/1/1999', '12-11-2024', '12/11/24'],
)

In [ ]:
# ✏️ #28 — 12-hour time like 9:05 AM (hour 1-12, minutes 00-59, uppercase AM/PM)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['9:05 AM', '12:59 PM', '1:00 PM'],
    should_not_match=['13:00 PM', '0:30 AM', '9:5 AM', '9:05 am', '9:05AM'],
)

In [ ]:
# ✏️ #29 — License plate: 3 capitals, optional dash, 3 digits
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['ABC-123', 'XYZ999'],
    should_not_match=['AB-123', 'ABC--123', 'abc-123', 'ABC-12'],
)

In [ ]:
# ✏️ #30 — SSN-style number 123-45-6789
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['123-45-6789'],
    should_not_match=['123456789', '12-345-6789', '123-45-678'],
)

In [ ]:
# ✏️ #31 — Extract 6-digit hex colours from CSS (not 5-digit or 7-letter ones)
# Hint: a \b after the 6 digits rejects #abcdefg
pattern = r""   # <- your pattern here
check_findall(pattern, 'color: #FF0000; bg: #00ff7f; x: #12345; y: #abcdefg', ['#FF0000', '#00ff7f'])

In [ ]:
# ✏️ #32 — Extract file names ending in .txt, .csv or .py
pattern = r""   # <- your pattern here
check_findall(pattern, 'Open data.csv, notes.txt and run.py; skip image.png', ['data.csv', 'notes.txt', 'run.py'])

In [ ]:
# ✏️ #33 — Extract double-quoted strings (keep the quotes)
# Hint: "[^"]*"
pattern = r""   # <- your pattern here
check_findall(pattern, 'He said "hello" and "bye".', ['"hello"', '"bye"'])

In [ ]:
# ✏️ #34 — Capture key=value pairs (findall returns tuples when there are groups)
pattern = r""   # <- your pattern here
check_findall(pattern, 'a=1 b=two c=3', [('a', '1'), ('b', 'two'), ('c', '3')])

In [ ]:
# ✏️ #35 — Reformat 555-123-4567 as (555) 123-4567
pattern = r""   # <- your pattern here
repl = ""      # <- replacement
check_sub(pattern, repl, 'Call 555-123-4567 now', 'Call (555) 123-4567 now')

In [ ]:
# ✏️ #36 — Turn "Last, First" into "First Last"
pattern = r""   # <- your pattern here
repl = ""      # <- replacement
check_sub(pattern, repl, 'Smith, John and Doe, Jane', 'John Smith and Jane Doe')

In [ ]:
# ✏️ #37 — Strip HTML tags
pattern = r""   # <- your pattern here
repl = ""      # <- replacement
check_sub(pattern, repl, '<p>Hello <b>world</b></p>', 'Hello world')

In [ ]:
# ✏️ #38 — Split into sentences on . ! or ? followed by whitespace
pattern = r""   # <- your pattern here
check_split(pattern, 'Hi there. How are you? Fine!', ['Hi there', 'How are you', 'Fine!'])

In [ ]:
# ✏️ #39 — A valid Python variable name
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['x', '_tmp', 'var2', 'CamelCase'],
    should_not_match=['2x', 'my-var', '', 'a b'],
)

In [ ]:
# ✏️ #40 — A signed number: optional sign, digits, optional .digits
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['5', '-3.14', '+0.5', '100'],
    should_not_match=['3.', '-.5', '--5', '5.5.5', ''],
)

In [ ]:
# ✏️ #41 — A UTC offset like +05:30 or -08:00 (hours up to 14)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['+05:30', '-08:00', '+14:00'],
    should_not_match=['+15:00', '05:30', '+5:30', '+05:60'],
)

In [ ]:
# ✏️ #42 — Card number: four groups of 4 digits separated by a space or dash
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['1234 5678 9012 3456', '1234-5678-9012-3456'],
    should_not_match=[
            '1234567890123456', '1234 5678 9012 345', '12345 678 9012 3456',
            '1234  5678 9012 3456',
        ],
)

In [ ]:
# ✏️ #43 — ⭐ Same card number, but the separator must be consistent
# Hint: capture the first separator and reuse it with \1
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['1234 5678 9012 3456', '1234-5678-9012-3456'],
    should_not_match=['1234 5678-9012 3456', '1234-5678 9012-3456', '1234 5678 9012'],
)

In [ ]:
# ✏️ #44 — Find ALL-CAPS words of 2+ letters (standalone)
pattern = r""   # <- your pattern here
check_findall(pattern, 'NASA and the FBI saw a UFO, not Bob.', ['NASA', 'FBI', 'UFO'])

In [ ]:
# ✏️ #45 — Detect a doubled letter (like the 'll' in hello)
pattern = r""   # <- your pattern here
check_search(
    pattern,
    should_find=['hello', 'aa', 'bookkeeper'],
    should_not_find=['abc', 'a b', 'abab'],
)

In [ ]:
# ✏️ #46 — Prices with a currency symbol ($, € or £)
pattern = r""   # <- your pattern here
check_findall(pattern, 'Tea $3.50, wine €12, cab £7.25, gum 1.00', ['$3.50', '€12', '£7.25'])

In [ ]:
# ✏️ #47 — Percentages (integer or decimal, then %)
pattern = r""   # <- your pattern here
check_findall(pattern, 'Up 5%, down 2.5%, flat 0%, 10 percent', ['5%', '2.5%', '0%'])

In [ ]:
# ✏️ #48 — Dates like 'Mar 5, 1990' (3-letter month names)
pattern = r""   # <- your pattern here
check_findall(pattern, 'Born Mar 5, 1990; moved Nov 12, 2020; Foo 1, 2000.', ['Mar 5, 1990', 'Nov 12, 2020'])

In [ ]:
# ✏️ #49 — Named groups first, last, age from 'John Smith (42)'
pattern = r""   # <- your pattern here
check_named(pattern, 'John Smith (42)',
            {'first': 'John', 'last': 'Smith', 'age': '42'})

In [ ]:
# ✏️ #50 — Named groups year, month, day, hour, minute from a timestamp
pattern = r""   # <- your pattern here
check_named(pattern, '2024-11-12T14:03',
            {'year': '2024', 'month': '11', 'day': '12', 'hour': '14', 'minute': '03'})

In [ ]:
# ✏️ #51 — Capture the score and the maximum from 'Rating: 4.5/5'
pattern = r""   # <- your pattern here
check_groups(pattern, 'Rating: 4.5/5', ('4.5', '5'))

In [ ]:
# ✏️ #52 — Lookahead: words followed by a colon (don't include the colon)
pattern = r""   # <- your pattern here
check_findall(pattern, 'name: Bob, age: 30, city Paris', ['name', 'age'])

In [ ]:
# ✏️ #53 — Lookbehind: digits that come right after ID-
pattern = r""   # <- your pattern here
check_findall(pattern, 'ID-42 and ID-7, but KEY-9', ['42', '7'])

In [ ]:
# ✏️ #54 — Negative lookbehind: standalone numbers NOT preceded by $
pattern = r""   # <- your pattern here
check_findall(pattern, '$5 and 10 and $20 and 3', ['10', '3'])

In [ ]:
# ✏️ #55 — Password: 10+ chars, no whitespace, at least one of ! @ # $
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['abcdefgh!j', 'Passw0rd#12'],
    should_not_match=['abcdefghij', 'short!', 'has space!!!!'],
)

In [ ]:
# ✏️ #56 — A binary byte: exactly eight 0/1 digits
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['00000000', '10101010', '11111111'],
    should_not_match=['1010101', '101010101', '1010101a', '10102010'],
)

In [ ]:
# ✏️ #57 — ⭐ Roman numerals up to 3999 (and not the empty string!)
# Hint: thousands, hundreds, tens, units. Each is a small alternation. Start with (?=.) so the empty string fails
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['I', 'IV', 'IX', 'XIV', 'MMXXIV', 'MCMXCIV', 'XLII'],
    should_not_match=['IIII', 'VX', 'IC', '', 'ABC', 'MMMM'],
)

In [ ]:
# ✏️ #58 — 24-hour time HH:MM with real ranges
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['00:00', '23:59', '09:30'],
    should_not_match=['24:00', '12:60', '9:30', '12:5'],
)

In [ ]:
# ✏️ #59 — MM/DD with real ranges (month 01-12, day 01-31)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['01/31', '12/25', '02/29'],
    should_not_match=['13/01', '00/10', '01/32', '1/5'],
)

In [ ]:
# ✏️ #60 — Pull the domain out of each email address (lookbehind for the @)
pattern = r""   # <- your pattern here
check_findall(pattern, 'a@x.com, b.c@mail.org; bad@', ['x.com', 'mail.org'])

---
# 🟠 Challenge (61–85)

In [ ]:
# ✏️ #61 — Semantic version: MAJOR.MINOR.PATCH with an optional -prerelease (letters, digits, dots)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['1.2.3', '10.0.12', '1.0.0-beta.1', '2.1.0-rc1'],
    should_not_match=['1.2', '1.2.3.4', 'v1.2.3', '1.2.3-', 'a.b.c'],
)

In [ ]:
# ✏️ #62 — A domain name: labels (alphanumeric, inner hyphens OK, not at the ends) + a 2+ letter TLD
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['example.com', 'sub.example.co.uk', 'a-b.org', 'x1.io'],
    should_not_match=['-bad.com', 'bad-.com', 'nodot', 'a..com', 'example.c', 'exa mple.com'],
)

In [ ]:
# ✏️ #63 — Flexible US phone: optional +1, area code with or without parens, optional space/dash separators
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=[
            '555-123-4567', '(555) 123-4567', '+1 555 123 4567', '5551234567',
            '+15551234567',
        ],
    should_not_match=['555-1234', '(555 123-4567', '+2 555 123 4567', '555-123-45678'],
)

In [ ]:
# ✏️ #64 — ISO date YYYY-MM-DD with real month/day ranges
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['2024-11-12', '1999-01-31', '2000-12-01'],
    should_not_match=['2024-13-12', '2024-00-10', '2024-11-32', '2024-11-00', '24-11-12'],
)

In [ ]:
# ✏️ #65 — Extract 12-hour times (with AM/PM, optional space) from text
pattern = r""   # <- your pattern here
check_findall(pattern, 'Open 9:00 AM to 5:30PM, closed at 13:00 or 7:5 PM.', ['9:00 AM', '5:30PM'])

In [ ]:
# ✏️ #66 — Parse a config line: key, then =, then value (ignore leading spaces and trailing comments)
# Hint: names: key and value
pattern = r""   # <- your pattern here
check_named(pattern, '  timeout =  30  # seconds',
            {'key': 'timeout', 'value': '30'})
check_named(pattern, 'name=alice',
            {'key': 'name', 'value': 'alice'})

In [ ]:
# ✏️ #67 — Extract URLs, but don't swallow trailing punctuation
# Hint: make the LAST character a \w so a trailing , or . is excluded
pattern = r""   # <- your pattern here
check_findall(pattern, 'Visit https://example.com/a-b, or http://foo.org/x.html. Not ftp://bar.com', ['https://example.com/a-b', 'http://foo.org/x.html'])

In [ ]:
# ✏️ #68 — Extract log levels from square brackets
pattern = r""   # <- your pattern here
check_findall(pattern, '[ERROR] x [INFO] y [ERROR] z', ['ERROR', 'INFO', 'ERROR'])

In [ ]:
# ✏️ #69 — Split a CSV line into fields, keeping quoted fields (which may contain commas) whole
# Hint: alternation: a quoted string OR a run of non-commas
pattern = r""   # <- your pattern here
check_findall(pattern, 'a,"b,c",d', ['a', '"b,c"', 'd'])

In [ ]:
# ✏️ #70 — camelCase → add an underscore between a lowercase and an uppercase letter
# Hint: a pattern can match an empty position between two characters
pattern = r""   # <- your pattern here
repl = ""      # <- replacement
check_sub(pattern, repl, 'getHttpResponseCode', 'get_Http_Response_Code')

In [ ]:
# ✏️ #71 — ⭐ Insert thousands separators
# Hint: a position that is inside a number and followed by groups of 3 digits up to the end of the number
pattern = r""   # <- your pattern here
repl = ""      # <- replacement
check_sub(pattern, repl, '1234567 and 89012 and 345', '1,234,567 and 89,012 and 345')

In [ ]:
# ✏️ #72 — Trim leading and trailing whitespace
pattern = r""   # <- your pattern here
repl = ""      # <- replacement
check_sub(pattern, repl, '   padded text \n', 'padded text')

In [ ]:
# ✏️ #73 — Mask all but the last 4 digits of a card number
# Hint: mask a digit if at least 4 more digits follow it (with any non-digits between)
pattern = r""   # <- your pattern here
repl = ""      # <- replacement
check_sub(pattern, repl, '1234 5678 9012 3456', '**** **** **** 3456')

In [ ]:
# ✏️ #74 — Numbers with optional minus, thousands commas and decimals
pattern = r""   # <- your pattern here
check_findall(pattern, 'Profit -1,234.50, loss 987, rate 0.75, big 12,345,678', ['-1,234.50', '987', '0.75', '12,345,678'])

In [ ]:
# ✏️ #75 — A lowercase UUID (8-4-4-4-12 hex digits)
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['123e4567-e89b-12d3-a456-426614174000'],
    should_not_match=[
            '123e4567e89b12d3a456426614174000', '123e4567-e89b-12d3-a456-42661417400',
            '123e4567-e89b-12d3-a456-42661417400g', '123e4567-e89b-12d3-a456',
        ],
)

In [ ]:
# ✏️ #76 — A Cisco-style MAC address: three groups of four hex digits separated by dots
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['001A.2B3C.4D5E', 'aabb.ccdd.eeff'],
    should_not_match=['001A.2B3C', '001A:2B3C:4D5E', '001A.2B3C.4D5E.6F70', '001A.2B3C.4D5G'],
)

In [ ]:
# ✏️ #77 — A hex-dump line: 8-digit lowercase hex offset, colon, then exactly 8 space-separated byte pairs
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=[
            '00000010: 48 65 6c 6c 6f 20 57 6f', '0000ff00: 00 00 00 00 00 00 00 00',
        ],
    should_not_match=[
            '00000010: 48 65 6c 6c 6f 20 57', '00000010 48 65 6c 6c 6f 20 57 6f',
            '00000010: 48 65 6c 6c 6f 20 57 6f 21', '0000010: 48 65 6c 6c 6f 20 57 6f',
        ],
)

In [ ]:
# ✏️ #78 — Extract hex address ranges like 0x1000-0x1FFF
pattern = r""   # <- your pattern here
check_findall(pattern, 'mapped 0x1000-0x1FFF and 0xdead-0xBEEF; bad 0x-0x', ['0x1000-0x1FFF', '0xdead-0xBEEF'])

In [ ]:
# ✏️ #79 — Extract ip:port pairs (loose IPv4, port 1-5 digits)
pattern = r""   # <- your pattern here
check_findall(pattern, 'connect 10.0.0.1:8080 and 192.168.1.5:443; not 10.0.0.1 or 1.2.3:80', ['10.0.0.1:8080', '192.168.1.5:443'])

In [ ]:
# ✏️ #80 — Parse an Apache access-log line into ip, date, method, path, status, size
# Hint: the date is everything inside [...]; the request is inside double quotes
pattern = r""   # <- your pattern here
check_named(pattern, '203.0.113.9 - - [12/Nov/2024:14:03:22 +0000] "GET /index.html HTTP/1.1" 200 5120',
            {'ip': '203.0.113.9', 'date': '12/Nov/2024:14:03:22 +0000', 'method': 'GET', 'path': '/index.html', 'status': '200', 'size': '5120'})

In [ ]:
# ✏️ #81 — Extract key=value pairs where the value may be "quoted with spaces"
pattern = r""   # <- your pattern here
check_findall(pattern, 'name="John Smith" age=30 city=Paris', [('name', '"John Smith"'), ('age', '30'), ('city', 'Paris')])

In [ ]:
# ✏️ #82 — A Python integer literal: decimal (no leading zeros), 0x hex, 0b binary, underscores between digits OK
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['0', '42', '1_000', '0xFF', '0xdead_beef', '0b1010', '000'],
    should_not_match=['01', '1__0', '_1', '0x', '0b102', '1_'],
)

In [ ]:
# ✏️ #83 — A compact duration like 1h30m15s: any of h, m, s present, in that order, at least one
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['1h', '30m', '15s', '1h30m', '1h30m15s', '2h15s'],
    should_not_match=['', 'h', '1h 30m', '30m1h', '1x'],
)

In [ ]:
# ✏️ #84 — Parse "Display Name <email>" into name and email (lazy name!)
pattern = r""   # <- your pattern here
check_named(pattern, 'Alice Smith <alice@example.com>',
            {'name': 'Alice Smith', 'email': 'alice@example.com'})

In [ ]:
# ✏️ #85 — Collapse accidentally repeated words (one or more repeats) down to one
pattern = r""   # <- your pattern here
repl = ""      # <- replacement
check_sub(pattern, repl, 'this is is a test test test', 'this is a test')

---
# 🔴 Boss (86–100)

In [ ]:
# ✏️ #86 — A PRIVATE IPv4 address: 10.x.x.x, 172.16-31.x.x, or 192.168.x.x (each octet 0-255)
# Hint: three alternatives joined with |; you're given `octet`
octet = r"(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)"
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=[
            '10.0.0.1', '10.255.255.255', '172.16.0.1', '172.31.255.255', '192.168.1.1',
        ],
    should_not_match=[
            '172.15.0.1', '172.32.0.1', '192.169.1.1', '11.0.0.1', '10.0.0.256', '8.8.8.8',
        ],
)

In [ ]:
# ✏️ #87 — A TCP port number 1-65535 (no leading zeros)
# Hint: split by length and leading digits, like the IPv4 octet
pattern = r""   # <- your pattern here
check_fullmatch(
    pattern,
    should_match=['1', '80', '443', '8080', '12345', '60000', '65535'],
    should_not_match=['0', '65536', '099', '70000', '-1', '100000', '655350'],
)

In [ ]:
# ✏️ #88 — Split a URL into scheme, host, port, path, query and frag (missing parts are None)
# Hint: everything after the host is optional: wrap each piece in (?:...)? or an optional named group
pattern = r""   # <- your pattern here
check_named(pattern, 'https://user:pw@example.com:8080/a/b?x=1#top',
            {'scheme': 'https', 'host': 'example.com', 'port': '8080', 'path': '/a/b', 'query': 'x=1', 'frag': 'top'})
check_named(pattern, 'http://localhost',
            {'scheme': 'http', 'host': 'localhost', 'port': None, 'path': None, 'query': None, 'frag': None})

In [ ]:
# ✏️ #89 — Tokenize an arithmetic expression (whitespace is skipped automatically)
# Hint: named groups NUM, ID, OP, LP, RP
pattern = r""   # <- your pattern here
text = '3.5*(x_1 + 42)-y'
check_tokens(pattern, text, [
        ('NUM', '3.5'), ('OP', '*'), ('LP', '('), ('ID', 'x_1'), ('OP', '+'),
        ('NUM', '42'), ('RP', ')'), ('OP', '-'), ('ID', 'y'),
    ])

In [ ]:
# ✏️ #90 — Tokenize a log line into DATE, LEVEL (ERROR|WARN|INFO), KV (key=value) and DURATION (like 123ms)
pattern = r""   # <- your pattern here
text = '2024-11-12 ERROR user=alice ip=10.0.0.7 took 123ms'
check_tokens(pattern, text, [
        ('DATE', '2024-11-12'), ('LEVEL', 'ERROR'), ('KV', 'user=alice'),
        ('KV', 'ip=10.0.0.7'), ('DURATION', '123ms'),
    ])

In [ ]:
# ✏️ #91 — Extract markdown links [text](url) as (text, url) pairs
pattern = r""   # <- your pattern here
check_findall(pattern, 'See [Python](https://python.org) and [Docs](https://docs.python.org/3/)', [
        ('Python', 'https://python.org'), ('Docs', 'https://docs.python.org/3/'),
    ])

In [ ]:
# ✏️ #92 — Wrap every URL in an HTML link: <a href="URL">URL</a>
# Hint: capture the URL, then use \1 twice in the replacement
pattern = r""   # <- your pattern here
repl = ""      # <- replacement
check_sub(pattern, repl, 'go to https://x.org now', 'go to <a href="https://x.org">https://x.org</a> now')

In [ ]:
# ✏️ #93 — Replacement can be a FUNCTION. It receives the Match object and returns the new text.
# Double every number: "3 apples and 10 pears" -> "6 apples and 20 pears"
# Fill in the pattern AND the function (hint: str(int(m.group()) * 2)).
pattern = r""   # <- your pattern here
repl = lambda m: m.group()      # <- e.g. lambda m: ...
check_sub(pattern, repl, "3 apples and 10 pears", "6 apples and 20 pears")
check_sub(pattern, repl, "7-14-21", "14-28-42")

In [ ]:
# ✏️ #94 — MULTILINE: find whole comment lines (lines that start with #)
# Hint: ^ and $ only match at line boundaries when you pass re.MULTILINE
pattern = r""   # <- your pattern here
flags = 0     # <- flags
check_findall(pattern, "# first\ncode = 1\n# second\nx = '#not a comment line'", ['# first', '# second'], flags)

In [ ]:
# ✏️ #95 — DOTALL: capture everything between BEGIN and END, even across lines
# Hint: . doesn't match newlines unless you pass re.DOTALL (and use a lazy .*?)
pattern = r""   # <- your pattern here
flags = 0     # <- flags
check_findall(pattern, 'x BEGIN\nline1\nline2\nEND y', ['\nline1\nline2\n'], flags)

### pandas round (96–100)

These use the pandas checkers from Part 12 of the main notebook. The cell below defines them again, plus the `logs` DataFrame.

In [ ]:
import pandas as pd

def _show_err(e):
    print(f"❌ pandas raised {type(e).__name__}: {e}")

def check_mask(pattern, series, expected, how="contains"):
    """Runs series.str.<how>(pattern) and compares the True/False list."""
    try:
        got = getattr(series.str, how)(pattern).tolist()
    except Exception as e:
        return _show_err(e)
    _compare(got, expected, how)

def check_replace(pattern, repl, series, expected):
    """Runs series.str.replace(pattern, repl, regex=True)."""
    try:
        got = series.str.replace(pattern, repl, regex=True).tolist()
    except Exception as e:
        return _show_err(e)
    _compare(got, expected, "replace")

def check_extract(pattern, series, expected):
    """Runs series.str.extract(pattern) and compares each named column."""
    try:
        got = series.str.extract(pattern).fillna("").to_dict("list")
    except Exception as e:
        return _show_err(e)
    _compare(got, expected, "extract")

def check_count(pattern, series, expected):
    """Runs series.str.count(pattern)."""
    try:
        got = series.str.count(pattern).tolist()
    except Exception as e:
        return _show_err(e)
    _compare(got, expected, "count")

def check_columns(df, expected):
    """Compare whole columns of a DataFrame (NaN/None are treated as '')."""
    problems = []
    for col, exp in expected.items():
        if col not in df.columns:
            problems.append(f"   missing column {col!r}")
            continue
        got = df[col].fillna("").astype(str).tolist()
        if got != exp:
            problems.append(f"   column {col!r}:\n      expected {exp}\n      got      {got}")
    _report(problems, len(expected))

logs = pd.DataFrame({"raw": [
    "2024-11-12 14:03:22 [ERROR] 192.168.1.15 GET /api/users?id=42 500 123ms",
    "2024-11-12 14:05:01 [INFO] 10.0.0.7 POST /login 200 87ms",
    "2024-11-12 14:07:45 [WARN] 192.168.1.15 GET /api/orders 404 15ms",
    "2024-11-12 14:09:10 [ERROR] 172.16.0.4 PUT /api/items/9 503 2210ms",
    "2024-11-12 14:11:30 [INFO] 10.0.0.7 GET /home 200 45ms",
]})
logs

In [ ]:
# ✏️ #96 — extract: named groups `name` and `age` from strings like "Alice (30)"
# (checker runs  people.str.extract(pattern) )
people = pd.Series(["Alice (30)", "Bob (25)", "Carol (41)"])
pattern = r""   # <- your pattern here
check_extract(pattern, people, {"name": ["Alice", "Bob", "Carol"], "age": ["30", "25", "41"]})

In [ ]:
# ✏️ #97 — count: how many words are in each string?
# (checker runs  s.str.count(pattern) )
s = pd.Series(["hello world", "one", "a b c d", ""])
pattern = r""   # <- your pattern here
check_count(pattern, s, [2, 1, 4, 0])

In [ ]:
# ✏️ #98 — fullmatch: which entries are valid email addresses?
# (checker runs  emails.str.fullmatch(pattern) )
emails = pd.Series(["a@b.com", "bad@", "x y@z.org", "ok.name+t@sub.example.co"])
pattern = r""   # <- your pattern here
check_mask(pattern, emails, [True, False, False, True], how="fullmatch")

In [ ]:
# ✏️ #99 — replace: trim the ends AND squeeze inner whitespace runs down to one space,
# using ONE pattern and an empty replacement.
# Hint: three alternatives:  leading spaces | trailing spaces | a space that is followed by another space
# (checker runs  s.str.replace(pattern, "", regex=True) )
s = pd.Series(["  a   b  ", "c    d", "e"])
pattern = r""   # <- your pattern here
check_replace(pattern, "", s, ["a b", "c d", "e"])

In [ ]:
# ✏️ #100 — extract: from the `logs` DataFrame, pull the FIRST path segment of each request
# (e.g. "/api/users?id=42" -> "api"). Name the group `endpoint`.
# Hint: the request starts after the HTTP method (GET|POST|PUT|DELETE) and a space, then a /
# (checker runs  logs["raw"].str.extract(pattern) )
pattern = r""   # <- your pattern here
check_extract(pattern, logs["raw"], {"endpoint": ["api", "login", "api", "api", "home"]})

---
# 🎉 Done!

If you got through these, you can read and write most real-world regex you'll meet. Next steps:

* Re-do any you needed the answer key for. **Cold, tomorrow.**
* Write your own: take any messy column in a dataset you're working with and try to parse it with `str.extract`.
* Use [regex101.com](https://regex101.com) (Python flavour) to see how the tricky ones (#57, #71, #73, #88) actually match, step by step.

---
# 🔑 Answer key

Try first. Your pattern can differ from mine and still be correct, as long as the checker is happy.

<details>
<summary><b>🔑 Answer key — click to open (try first!)</b></summary>

<pre>
1      r"\d"
2      r"\d+"
3      r"the"
4      r"\$5\.00"
5      r"b.t"
6      r"\d{4}"
7      r"[A-Z]{2}"
8      r"[A-Z][a-z]+"
9      r"\d{5}(?:-\d{4})?"
10     r"[a-z]+"
11     r"\s+"
12     r"\S+"
13     r"y(?:es)?|no?"
14     r"\b\w+ing\b"
15     r"[0-9a-fA-F]+"
16     r"-?\d+"
17     r"\d+\.\d+"
18     r"\d{2}:\d{2}"
19     r"#\w+"
20     r"@\w+"
21     r"[01]+"
22
       P = r"\d"
       R = r"X"
23     r",\s*"
24     r"^[A-Z]"
25     r"\?$"
26     r"\d{3}-\d{3}-\d{4}"
27     r"\d{2}/\d{2}/\d{4}"
28     r"(?:[1-9]|1[0-2]):[0-5]\d (?:AM|PM)"
29     r"[A-Z]{3}-?\d{3}"
30     r"\d{3}-\d{2}-\d{4}"
31     r"#[0-9a-fA-F]{6}\b"
32     r"\w+\.(?:txt|csv|py)"
33     r'"[^"]*"'
34     r"(\w+)=(\w+)"
35
       P = r"(\d{3})-(\d{3})-(\d{4})"
       R = r"(\1) \2-\3"
36
       P = r"(\w+), (\w+)"
       R = r"\2 \1"
37
       P = r"&lt;[^&gt;]+&gt;"
       R = r""
38     r"[.!?]\s+"
39     r"[A-Za-z_]\w*"
40     r"[+-]?\d+(?:\.\d+)?"
41     r"[+-](?:0\d|1[0-4]):[0-5]\d"
42     r"\d{4}(?:[ -]\d{4}){3}"
43     r"\d{4}([ -])\d{4}\1\d{4}\1\d{4}"
44     r"\b[A-Z]{2,}\b"
45     r"(\w)\1"
46     r"[$€£]\d+(?:\.\d{2})?"
47     r"\d+(?:\.\d+)?%"
48     r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{1,2}, \d{4}"
49     r"(?P&lt;first&gt;\w+) (?P&lt;last&gt;\w+) \((?P&lt;age&gt;\d+)\)"
50     r"(?P&lt;year&gt;\d{4})-(?P&lt;month&gt;\d{2})-(?P&lt;day&gt;\d{2})T(?P&lt;hour&gt;\d{2}):(?P&lt;minute&gt;\d{2})"
51     r"(\d+(?:\.\d+)?)/(\d+)"
52     r"\w+(?=:)"
53     r"(?&lt;=ID-)\d+"
54     r"(?&lt;!\$)\b\d+\b"
55     r"(?=.*[!@#$])\S{10,}"
56     r"[01]{8}"
57     r"(?=.)M{0,3}(?:CM|CD|D?C{0,3})(?:XC|XL|L?X{0,3})(?:IX|IV|V?I{0,3})"
58     r"(?:[01]\d|2[0-3]):[0-5]\d"
59     r"(?:0[1-9]|1[0-2])/(?:0[1-9]|[12]\d|3[01])"
60     r"(?&lt;=@)[\w.-]+"
61     r"\d+\.\d+\.\d+(?:-[A-Za-z0-9.]+)?"
62     r"(?:[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?\.)+[A-Za-z]{2,}"
63     r"(?:\+1[ -]?)?(?:\(\d{3}\)|\d{3})[ -]?\d{3}[ -]?\d{4}"
64     r"\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])"
65     r"\b(?:[1-9]|1[0-2]):[0-5]\d\s?[AP]M\b"
66     r"^\s*(?P&lt;key&gt;\w+)\s*=\s*(?P&lt;value&gt;\S+)"
67     r"https?://[\w./-]*\w"
68     r"\[(\w+)\]"
69     r'"[^"]*"|[^,]+'
70
       P = r"(?&lt;=[a-z])(?=[A-Z])"
       R = r"_"
71
       P = r"\B(?=(?:\d{3})+(?!\d))"
       R = r","
72
       P = r"^\s+|\s+$"
       R = r""
73
       P = r"\d(?=(?:\D*\d){4})"
       R = r"*"
74     r"-?\d{1,3}(?:,\d{3})*(?:\.\d+)?"
75     r"[0-9a-f]{8}(?:-[0-9a-f]{4}){3}-[0-9a-f]{12}"
76     r"(?:[0-9a-fA-F]{4}\.){2}[0-9a-fA-F]{4}"
77     r"[0-9a-f]{8}: (?:[0-9a-f]{2} ){7}[0-9a-f]{2}"
78     r"0x[0-9A-Fa-f]+-0x[0-9A-Fa-f]+"
79     r"\b\d{1,3}(?:\.\d{1,3}){3}:\d{1,5}\b"
80     r'(?P&lt;ip&gt;\S+) \S+ \S+ \[(?P&lt;date&gt;[^\]]+)\] "(?P&lt;method&gt;[A-Z]+) (?P&lt;path&gt;\S+) [^"]*" (?P&lt;status&gt;\d{3}) (?P&lt;size&gt;\d+)'
81     r'(\w+)=("[^"]*"|\S+)'
82     r"0[xX][0-9a-fA-F]+(?:_[0-9a-fA-F]+)*|0[bB][01]+(?:_[01]+)*|[1-9]\d*(?:_\d+)*|0+"
83     r"(?=.)(?:\d+h)?(?:\d+m)?(?:\d+s)?"
84     r"(?P&lt;name&gt;[^&lt;]+?)\s*&lt;(?P&lt;email&gt;[^&gt;]+)&gt;"
85
       P = r"\b(\w+)(?:\s+\1\b)+"
       R = r"\1"
86     r"(?:10(?:\." + octet + r"){3}|172\.(?:1[6-9]|2\d|3[01])(?:\." + octet + r"){2}|192\.168(?:\." + octet + r"){2})"
87     r"[1-9]\d{0,3}|[1-5]\d{4}|6[0-4]\d{3}|65[0-4]\d{2}|655[0-2]\d|6553[0-5]"
88     r"(?P&lt;scheme&gt;https?)://(?:\S+@)?(?P&lt;host&gt;[\w.-]+)(?::(?P&lt;port&gt;\d+))?(?P&lt;path&gt;/[^?#\s]*)?(?:\?(?P&lt;query&gt;[^#\s]*))?(?:#(?P&lt;frag&gt;\S*))?"
89     r"(?P&lt;NUM&gt;\d+(?:\.\d+)?)|(?P&lt;ID&gt;[A-Za-z_]\w*)|(?P&lt;OP&gt;[-+*/^])|(?P&lt;LP&gt;\()|(?P&lt;RP&gt;\))"
90     r"(?P&lt;DATE&gt;\d{4}-\d{2}-\d{2})|(?P&lt;LEVEL&gt;ERROR|WARN|INFO)|(?P&lt;KV&gt;\w+=\S+)|(?P&lt;DURATION&gt;\d+ms)"
91     r"\[([^\]]+)\]\(([^)]+)\)"
92
       P = r"(https?://\S+)"
       R = r'&lt;a href="\1"&gt;\1&lt;/a&gt;'
93
       P = r"\d+"
       R = lambda m: str(int(m.group()) * 2)
94
       P = r"^#.*$"
       F = re.MULTILINE
95
       P = r"BEGIN(.*?)END"
       F = re.DOTALL
96     r"(?P&lt;name&gt;\w+) \((?P&lt;age&gt;\d+)\)"
97     r"\b\w+\b"
98     r"[\w.+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+"
99     r"^\s+|\s+$|\s+(?=\s)"
100    r"(?:GET|POST|PUT|DELETE) /(?P&lt;endpoint&gt;\w+)"
</pre>

</details>